In [ ]:
# 1а. Завантажуємо необхідні бібліотеки для виконання дослідницького аналізу даних

import pandas as pd
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, Table, Column, Integer, String, MetaData
from sqlalchemy.engine import URL

In [ ]:
# 1b. З'єднання з базою PostgreSQL та зчитування необхідних даних

# ф-ція з'єднання з базою PostgreSQL
def get_db_connection(
    db_host: str,
    db_name: str,
    db_user: str,
    db_password: str,
    db_port: int,
):
    url = URL.create(
        drivername="postgresql+psycopg",
        username=db_user,
        password=db_password,
        host=db_host,
        port=db_port,
        database=db_name,
    )
    engine = create_engine(url, pool_pre_ping=True, pool_recycle=3600, echo=False)
    return engine

# завантажуємо змінні з файлу .env: 
load_dotenv(dotenv_path='.env')

# привласнюємо відповідні значення:
db_host = os.getenv("POSTGRES_HOST")
db_name = os.getenv("POSTGRES_DB")
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
db_port = os.getenv("POSTGRES_PORT")

# спроба з'єднання з базою PostgreSQL
try:
    engine = get_db_connection(
        db_host=db_host,
        db_name=db_name,
        db_user=db_user,
        db_password=db_password,
        db_port=db_port,
    )
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
except Exception as ex:
    print(
        "Connection could not be made due to the following error:\n", ex
    )  # -повідомлення у разі невдалого з'єднання
    # аварійно завершуємо програму
    exit(1)

# зчитування необхідних даних:
period = pd.read_sql("SELECT * FROM data_period", engine)
year_stat = pd.read_sql("SELECT * FROM year_stat", engine)
    
# налаштуємо додтаткові параметри для коректного відображення завантажених даних:
pd.set_option('display.max_colwidth', None)
pd.options.display.width = 170
year_stat['years'] = year_stat['years'].astype('int64')
year_stat.set_index('years', inplace=True)
year_stat.index.name = None

In [ ]:
# для завантажених вхідних даних, у файлі 'period', стовбчик ['record_date'] перетворимо у тип даних для роботи з датою та часом - "datetime64":
period['record_date'] = period['record_date'].astype('datetime64[ns]')

In [ ]:
# 2. Для коректності подальших розрахунків зазначимо межі існування деяких категорій

period_c = period.copy()
# замінимо всі пусті комірки - "NaN" нулями -"0" та представимо усі значення цілими числами:
for x in period_c.iloc[:, 2:]:
  period_c[x] = period_c[x].astype('float').fillna(0)
  period_c[x] = period_c[x].astype('int')
# за межами існування нижчезазначених категорій замінимо "0" на "NaN":
period_c.loc[period_c.record_date <= '2013.03.01', 'health'] = np.nan # (2013.03.01 - 2017.04.30)
period_c.loc[period_c.record_date <= '2013.03.01', 'household_goods'] = np.nan # (2013.03.01 - 2017.04.30)
period_c.loc[period_c.record_date >= '2016.01.01', 'debts'] = np.nan # (2013.01.01 - 2015.12.31)
period_c.loc[period_c.record_date < '2016.01.01', 'clothes_shoes'] = np.nan # (2016.01.01 - 2017.04.30)

In [ ]:
# 3 АНАЛІТИКА НАДХОДЖЕНЬ/ВИТРАТ НА РІВНІ - 'РІК'

In [ ]:
# 3a. Побудуємо графіки (3.1.1 - 3.1.3) на основі розрахованих показників для аналізу даних на рівні - "Рік"

# Вхідні дані:
years = year_stat.index
x3 = np.arange(len(years))
y1_1 = year_stat['inc_sum'].values
y2_1 = year_stat['exp_sum'].values
y3_1 = year_stat['inc_mean'].values
y3_2 = year_stat['exp_mean'].values
y3_3 = year_stat['profit_mean'].values

colors_1 = [(0 , 0.6, 0, 0.55), (0 , 0.6, 0, 0.4), (0 , 0.6, 0, 0.45), (0 , 0.6, 0, 0.7)] # -відтінки зеленого у форматі (R, G, B, Alpha)
colors_2 = [(0.9 , 0, 0, 0.53), (0.9 , 0, 0, 0.35), (0.9 , 0, 0, 0.28), (0.9 , 0, 0, 0.6)] # -відтінки червоного у форматі (R, G, B, Alpha)

fig = plt.figure(figsize=(13, 8)) # -задаємо Розмір графіків

# Перший графік:
ax1 = plt.subplot2grid((2, 2), (0, 0)) # -розташування у першому рядку, першому стовбчику
ax1.pie(y1_1, 
        labels=years, 
        colors=colors_1, 
        wedgeprops={'edgecolor': 'black', 'linewidth':0.4}, # -параметри контурних ліній
        autopct='%1.1f%%', 
        counterclock=False, # -розташування за часовою стрілкою
        startangle=90)
# Оформлення:
ax1.set_title('3.2.1 Доходи') # -назва графіка

# Другий графік:
ax2 = plt.subplot2grid((2, 2), (0, 1)) # -розташування у першому рядку, другому стовбчику
ax2.pie(y2_1,
        labels=years,
        colors=colors_2,
        wedgeprops={'edgecolor': 'black', 'linewidth':0.4},
        autopct='%1.1f%%',
        counterclock=False,
        startangle=90)
# Оформлення:
ax2.set_title('3.2.2 Витрати')

# Третій графік:
ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2) # -розташування у другому рядку, на обидва стовбчики
width=0.08
ax3.bar(x3-width/2, y3_1, width, label='Доходи', color='green', alpha=0.8, zorder=2) # -параметри візуалізації Доходів
ax3.bar(x3+width/2, y3_2, width, label='Витрати', color='red', alpha=0.8, zorder=2) # -параметри візуалізації Витрат
ax3.plot(x3, y3_3, label='Прибуток', color='orange', linewidth=3.5, alpha=0.8, zorder=2) # -параметри візуалізації Прибутку
# Оформлення:
ax3.set_title('3.2.3 Середнє для річних показників Бюджету') 
ax3.set_xticks(x3) # -визначення розташування поділів для осі "x"
ax3.set_xticklabels(years) # -явне присвоювання підписів для поділів на осі "x"
ax3.set_ylabel('Діапазон значень_грн') # -підпис осі "y"
ax3.legend(bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0., fontsize='small') # -відображення легенди зліва за межами графіку
ax3.grid(axis='y', linestyle='--', linewidth=0.35) # -параметри сітки
plt.ioff() # -вимикаємо НЕ явне відображення графіків

# 3b. Роздрукуємо візуалізації для аналізу для аналізу даних на рівні - "Рік":
plt.show()
print(' ')

In [ ]:
# ПІДГОТОВКА ДАНИХ ДЛЯ АНАЛІЗУ НА РІВНІ - "МІСЯЦЬ"

In [ ]:
# Для аналітики "Транзакцій на рівні - Місяць", агрегуємо вхідні дані за весь період:

# створимо нову таблицю, що містить лише Категоріальні дані за кожен День: 
day = period.iloc[:, 2:]
# з колонки "record_date" виокремлюємо стовбчик з назвами місяця - "month":
day['month'] = period['record_date'].dt.month_name()
# також додаємо стовбчик з роками - "year":
day['year'] = period['record_date'].dt.year

# day - "Проміжна таблиця Транзакцій" за весь період, для подальших розрахунків та аналізу даних на рівні - "Місяць"

# сгрупуємо дані по місяцях для кожного року:
sum_month = day.groupby(['year', 'month'], sort=False, as_index=False).sum()

# для коректності подальших розрахунків зазначимо межі існування деяких категорій:
sum_month.loc[sum_month.index <= 1, 'health'] = pd.NA # (2013.03.01 - 2017.04.30)
sum_month.loc[sum_month.index <= 1, 'household_goods'] = pd.NA # (2013.03.01 - 2017.04.30)
sum_month.loc[sum_month.year >= 2016, 'debts'] = pd.NA # (2013.01.01 - 2015.12.31)
sum_month.loc[sum_month.year < 2016, 'clothes_shoes'] = pd.NA # (2016.01.01 - 2017.04.30)

In [ ]:
# Здійснемо розрахунок "Відсотка від суми надходжень/витрат за Місяць", який містить кожна категорія.

# загальна сума Надходжень/Витрат за кожен місяць:
inc_sum_mo = sum_month.iloc[:, 2:4].sum(axis=1) # -використаємо "Агреговану таблицю Транзакцій" (п. 2a)
exp_sum_mo = sum_month.iloc[:, 4:].sum(axis=1)
# діапазон значень для категорій Надходжень/Витрат:
inc_mo = sum_month.iloc[:, 2:4]
exp_mo = sum_month.iloc[:, 4:]
# розрахуємо, яку частку складає кожна прибуткова/витратна категорія: 
share_inc = inc_mo.div(inc_sum_mo, axis=0)*100
share_exp = exp_mo.div(exp_sum_mo, axis=0)*100
# додамо відповідні значення років та місяців та поєднаємо все у спільну таблицю:
share_month = pd.concat([sum_month.iloc[:, :2], share_inc, share_exp], axis=1)

In [ ]:
# Для аналітики "Активності категорій на рівні - Місяць", перетворимо та сгрупуємо дані по місяцях для кожного року.

# виявимо активні категорії для кожного Дня:
active = period.iloc[:, 1:].groupby('record_date', as_index=False).count()
# видаляємо стовбчик з датами - "record_date":
act_mo = active.drop(columns=['record_date'])
# додаємо стовбчик з назвами місяця - "month":
act_mo['month'] = period['record_date'].dt.month_name()
# додаємо стовбчик з роками - "year":
act_mo['year'] = period['record_date'].dt.year

# act_mo - "Проміжна таблиця Активності" за весь період, для подальших розрахунків та аналізу даних на рівні - "Місяць"

# сгрупуємо дані по місяцях для кожного року:
active_month = act_mo.groupby(['year', 'month'], sort=False, as_index=False).sum()

# для коректності подальших розрахунків зазначимо межі існування деяких категорій:
active_month.loc[active_month.index <= 1, 'health'] = pd.NA # (2013.03.01 -  2017.04.30)
active_month.loc[active_month.index <= 1, 'household_goods'] = pd.NA # (2013.03.01 - 2017.04.30)
active_month.loc[active_month.year >= 2016, 'debts'] = pd.NA # (2013.01.01 - 2015.12.31)
active_month.loc[active_month.year < 2016, 'clothes_shoes'] = pd.NA # (2016.01.01 - 2017.04.30)

In [ ]:
# Для первинного дослідження місячної структури даних, проаналізуємо усі категорії Надходжень та Витрат разом.

# використаємо вже обчислені сумарні значення Надходжень/Витарт за кожен місяць (п. 2b), та змінимо назви стовбчиків:
inc_month = inc_sum_mo.copy().rename('inc_month')
exp_month = exp_sum_mo.copy().rename('exp_month')

# розрахуємо "Чистий прибуток" за кожен місяць:
profit_month = (inc_month - exp_month).rename('profit')

# виокремимо необхідний діапазон Активності Витрат:
expenses_active = active.iloc[:, 3:]
# задамо функцію, яка розрахує Наявність будь-яких Витрат для стовбця "active_exp_days":
def active_exp_days(x):
    """
    Input: "x" - кожне значення усіх рядків таблиці з Активностями Витрат
    Return: "1" - якщо хоч одне значення у рядку НЕ нульове/відсутнє
            "0" - якщо ВСІ значення у рядку нульові/відсутні
    """
    if (x == 0).all() == False:
        return 1
    else:
         return 0
# додаємо до результуючої таблиці стовбець - "active_exp_days", значення якого визначаться згідно діапазону категорій Витарт, за допомогою функції:
expenses_active['active_exp_days'] = expenses_active.apply(active_exp_days, axis=1)
# додаємо до результуючої таблиці стовбець - "zero_exp_days", який покаже наявність днів без Витрат:
expenses_active['zero_exp_days'] = 0
# змінемо значення нового стовбця за умовою: якщо НЕ було активних Витратних транзакцій, тоді виводимо 1:
expenses_active.loc[expenses_active.active_exp_days == 0, 'zero_exp_days'] = 1
# додаємо стовбчики з назвами місяця - "month" та року - "year":
expenses_active['month'] = active['record_date'].dt.month_name()
expenses_active['year'] = active['record_date'].dt.year

# expenses_active - "Проміжна таблиця деяких показників Витрат" для розрахунків та аналізу даних на рівні - "Місяць"

# сгрупуємо розраховані дані Ативностей по місяцях для кожного року:
exp_act_mo = expenses_active.groupby(['year', 'month'], sort=False, as_index=False).sum()
# об'єднаємо всі необхідні дані у спільну таблицю:
budget_month = pd.concat([exp_act_mo['year'], exp_act_mo['month'], inc_month, exp_month, profit_month, exp_act_mo.iloc[:, -2:]], axis=1)

# значення стовбчика 'month' робимо індексами:
budget_month.set_index('month', inplace=True)

# видаляємо заголовок індексів та змінюємо тип числових значень:
budget_month.index.name = None
budget_month[['inc_month', 'exp_month', 'profit']] = budget_month[['inc_month', 'exp_month', 'profit']].astype('float64')

# budget_month - "Агрегована таблиця Надходжень/Витрат за кажен Місяць"

In [ ]:
# "ТИПОВИЙ" МІСЯЦЬ

In [ ]:
# 4a. Проведемо аналітику узагальнених місячних даних за весь період.

# 1) "Медіана", "Середнє" та "Ст. відхилення" для значень "Агрегованої таблиці Надходжень/Витрат":
char_month = round(budget_month.iloc[:, 1:].agg(['median', 'mean', 'std']), 2).T 

# 2) "10 і 90 перцентилі" агрегованоих показників Надходжень/Витрат, щоб нівелювати вплив екстремальних значень на аналіз:
p10 = budget_month.iloc[:, 1:].quantile(0.1).rename('P10')
p90 = budget_month.iloc[:, 1:].quantile(0.9).rename('P90')

# 3) "Коефіцієнт варіації":
char_month['cv'] = char_month['std'].div(char_month['mean'])

# 4) об'єднаємо всі необхідні дані у спільну таблицю:
stat_month = pd.concat([char_month['median'], p10, p90, char_month[['mean','cv']]], axis=1)
# змінимо назви індексів та зменшимо кількість знаків після коми:
stat_month_ = round(stat_month.rename({'inc_month': 'Надходження:', 'exp_month': 'Витрати:', 'profit': 'Прибуток:', 
                             'active_exp_days': 'Активні витратні дні:', 'zero_exp_days': 'Тихі витратні дні:'}), 1)

# 5) "Медіанне значення Норми заощаджень":
saving_rate_med = round((1 - budget_month['exp_month'].median() / budget_month['inc_month'].median())*100, 2)

# 6) "Частка місяців з Дефіцитом":
pct_def = round(budget_month['profit'][budget_month['profit'] < 0].count() / budget_month['profit'].count()*100, 1)

In [ ]:
# 4b. Побудуємо графіки на основі розрахованих показників для подальшого аналізу "Типового" місяця:

# створення набору даних
data_budget = budget_month[['inc_month', 'exp_month', 'profit']].copy()
data_budget.rename(columns={'inc_month': 'Доходи', 'exp_month': 'Витрати', 'profit': 'Прибуток'}, inplace=True)

# налаштування візуалізації
plt.figure(figsize=(12, 5))

# побудова box plot
ax = sns.boxplot(data=data_budget, showmeans=True, palette="Set2", 
            meanprops={
                "marker": "o", 
                "markerfacecolor": "white", 
                "markeredgecolor": "black", 
                "markersize": "8"
            } # Налаштування зовнішнього вигляду маркера
) 

# оформлення
plt.title('4.2.1 Розподіл основних показників')
plt.ylabel('Сума_грн', fontsize='10')
ax.tick_params(axis='both', labelsize=10) # зменшуємо розмір значень на обох осях
plt.grid(axis='y')

plt.ioff()

# 4c. Роздрукуємо статистичні показники для аналізу "Типового" місяця та зробимо висновки:

print('\n4.1 "ТИПОВИЙ" МІСЯЦЬ:\n')
print(stat_month_, '\n')
print('Медіана Норми заощаджень:', saving_rate_med, '%')
print('Частка місяців з Дефіцитом:', pct_def, '%\n')
print('4.2 ВІЗУАЛІЗАЦІЇ:\n')
print("-4.2.1 'Розподіл основних показників'\n")
plt.show()
print('4.3 ВИСНОВКИ:\n')
print('1) Типовий місяць в', 100 - pct_def, '% випадків мав "профіцитний" бюджет, при цьому Грошовий потік дорівнював -', saving_rate_med, '% Норми заощаджень.')
print('2) Доходи та Розходи носили асиметричний розподіл даних, більшість з яких знаходилися на нижній межі можливих показників.')
print('3) Надходження складали -', int(stat_month.iat[0, 0]), 'грн/міс та значно коливалися у діапазоні:', int(stat_month.iat[0, 1]), '-',  int(stat_month.iat[0, 2]), 'грн/міс. Це можна пояснити одноразовими виручками або преміями.')
print('4) Витрати становили -', int(stat_month.iat[1, 0]), 'грн/міс і мали середню варіативність:', int(stat_month.iat[1, 1]), '-',  int(stat_month.iat[1, 2]), 'грн/міс. Місяць був дуже активним та складався з', int(stat_month.iat[3, 0]), 'Витратних днів.')
print('5) Типовий Прибуток мав аномально високу неоднорідність даних: cv =', round(stat_month.iat[2, 4], 1), ', а отже і дуже великий діапазон значень, від:', int(stat_month.iat[2, 1]), 'до:',  int(stat_month.iat[2, 2]), 'грн/міс.')
print('   Такий нестабільний характер Грошового потоку міг бути спричинений нерегулярними Доходами та наявністю рідких великих Розходів.\n\n')

In [ ]:
# КАТЕГОРІАЛЬНИЙ ПРОФІЛЬ МІСЯЦЯ

In [ ]:
# 5a. Проведемо аналітику даних за весь період, для кожної "Категорії на рівні - Місяць".

# 1) визначимо основні статистичні показники:
sum_cat = sum_month.iloc[:, 2:] # -використаємо "Агреговану таблицю Транзакцій" (п. 2a)
desc_cat = sum_cat.describe()
desc_cat = desc_cat.T

# 2) "Медіанне місячне значення Відсотка категорії від загальної суми надходжень/витрат":
share_cat = share_month.iloc[:, 2:] # -використаємо "Агреговану таблицю Відсотка категорій від Місячних надходжень/витрат" (п. 2b)
share_cat_med = share_cat.median()
share_cat_med_ = round(share_cat_med, 1)

# 3) "Медіанне місячне значення Активності":
active_cat = active_month.iloc[:, 2:] # -використаємо "Агреговану таблицю Активності за кожен Місяць" (п. 2c)
active_cat_med = active_cat.median()

# 4) "Коефіцієнт варіації":
desc_cat['cv'] = desc_cat['std'].div(desc_cat['mean'])
desc_cat = round(desc_cat, 1)

# 5) "Частка появи" кожної категорії (% місяців):
interval_cat = active_month.iloc[:, 2:].count()
count_cat = active_month.iloc[:, 2:].replace(0, np.nan).count()
#
appear_cat = count_cat.div(interval_cat, axis=0)*100
appear_cat_ = round(appear_cat, 1)

# 6) об'єднаємо всі необхідні дані у спільну таблицю:
stat_cat_month = pd.concat([desc_cat['50%'].rename('median'), share_cat_med_.rename('share_med_%'), 
                          active_cat_med.rename('activity_med'), desc_cat['mean'], desc_cat['cv'], 
                          desc_cat['min'], desc_cat['25%'].rename('P25'), desc_cat['75%'].rename('P75'), 
                          desc_cat['max'], appear_cat_.rename('appear_%')], axis=1)

In [ ]:
# 5b. Побудуємо графіки на основі розрахованих показників для категоріального аналізу на рівні - Місяць

# вхідні дані Доходів:
inc_cat = stat_cat_month.iloc[:2, :]
#
x = inc_cat.index
y1_1 = inc_cat['median'].values
y1_2 = inc_cat['mean'].values
y1_3 = inc_cat['share_med_%'].values
y1_4 = inc_cat['appear_%'].values

# вхідні дані Витрат:
exp_cat1 = stat_cat_month.iloc[2:, :].sort_values(by=['median'], ascending=False) # -сортування категорій для наочного представлення
index_exp1 = exp_cat1.index
exp_cat2 = stat_cat_month.iloc[2:, :].sort_values(by=['activity_med'], ascending=False)
index_exp2 = exp_cat2.index
#
x2_1 = np.arange(len(index_exp1))
x2_2 = np.arange(len(index_exp2))  
y2_1 = exp_cat1['median'].values
y2_2 = exp_cat1['mean'].values
y2_3 = exp_cat2['activity_med'].values
y2_4 = exp_cat2['appear_%'].values

fig = plt.figure(figsize=(12, 8))

# Перший графік:
ax1 = plt.subplot2grid((2, 2), (0, 0)) # -розташування у першому рядку, першому стовбчику
ax1.bar(x, y1_1, width = 0.3, label='median', color='seagreen', zorder=2) # -параметри візуалізації медіани категорій Доходів
ax1.bar(x, y1_2, width = 0.3, label='mean', color='lightseagreen', zorder=2, alpha=0.5) # -параметри візуалізації середнього категорій Доходів
# Оформлення:
ax1.set_title('5.2.1 Медіана vs Середнє категорій Доходів')
ax1.set_ylabel("Діапазон доходів_грн") # -підпис осі y
ax1.set_yticks(np.arange(0, 3000, 250)) # -діапазон та шаг значень вісі "y"
ax1.grid(axis='y', linestyle='--', linewidth=0.5) 
ax1.legend(loc='upper right', fontsize='small') # -показати легенду зверху у правому куті 

# Другий графік:
ax2 = plt.subplot2grid((2, 2), (0, 1)) # -розташування у першому рядку, другому стовбчику
ax2.bar(x2_1, y2_1, label='median', color='blue', zorder=2) # -параметри візуалізації медіани категорій Розходів
ax2.bar(x2_1, y2_2, label='mean', color='tab:blue', zorder=2, alpha=0.5) # -параметри візуалізації середнього категорій Розходів
# Оформлення:
ax2.set_title('5.2.2 Медіана vs Середнє категорій Витрат')
ax2.set_xticks(x2_1)
ax2.set_xticklabels(index_exp1, fontsize='small')
ax2.tick_params(axis='x', labelrotation=70) # -нахил значень вісі "х"
ax2.set_ylabel("Діапазон витрат_грн")
ax2.set_yticks(np.arange(0, 500, 50))
ax2.grid(axis='y', linestyle='--', linewidth=0.5)
ax2.legend(loc='upper right', fontsize='small')
# Виокремлення найбільш/найменш значущіх категорій:
for i, label in enumerate(ax2.get_xticklabels()):
    if i in x2_1[:2]: 
        label.set_color('green')
        label.set_fontweight('bold')
        label.set_fontsize(9.5)
    if i in x2_1[-3:]:
        if i == x2_1[-3]:
            if not (stat_cat_month.iloc[stat_cat_month.index.get_loc(index_exp1[-1])].isna()).all():
                continue
            else:
                label.set_color('red')
                label.set_fontweight('bold')
                label.set_fontsize(9.5)
        elif i == x2_1[-1]:
            if (stat_cat_month.iloc[stat_cat_month.index.get_loc(index_exp1[-1])].isna()).all():
                continue
            else:
                label.set_color('red')
                label.set_fontweight('bold')
                label.set_fontsize(9.5)
        else:
            label.set_color('red')
            label.set_fontweight('bold')
            label.set_fontsize(9.5)

# Третій графік:
ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2) # -розташування у другому рядку, на обидва стовбчики
width = 0.45
ax3.bar(x2_2 - width/2, y2_3, width, label='Активність', color='lightsalmon', zorder=2) # -параметри візуалізації Активності категорій
ax4 = ax3.twinx() # -створення додтакової вісі "y"
ax4.bar(x2_2 + width/2, y2_4, width, label='Ймовірність появи', color='plum', zorder=2) # -параметри візуалізації Ймовірності появи категорій
# Оформлення:
ax3.set_title('5.2.3 Частотні показники категорій Витрат')
ax3.set_xticks(x2_2)
ax3.set_xticklabels(index_exp2, fontsize='small')
ax3.tick_params(axis='x', labelrotation=35) # -нахил значень вісі "х"
ax3.set_yticks(np.arange(0, 19.5, 1.5)) # -діапазон та шаг значень основної вісі "y"
ax3.set_ylabel("Місячна активність_к-ть") # -підпис основної вісі "y"
ax3.legend(bbox_to_anchor=(-0.04, 1.12), loc='upper left', borderaxespad=0., fontsize='small')  # -відображення легенди за межами основної вісі "y"
ax3.grid(axis='y', linestyle='--', linewidth=0.5)
ax4.set_yticks(np.arange(0, 105, 10)) # -діапазон та шаг значень додаткової вісі "y"
ax4.set_ylabel("Ймовірність появи_%") # -підпис додаткової вісі "y"
ax4.legend(bbox_to_anchor=(1.03, 1.12), loc='upper right', borderaxespad=0., fontsize='small')
# Виокремлення найбільш/найменш значущіх категорій:
for i, label in enumerate(ax3.get_xticklabels()):
    if i in x2_2[:2]: 
        label.set_color('green')
        label.set_fontweight('bold')
        label.set_fontsize(9.5)
    if i in x2_2[-3:]:
        if i == x2_2[-3]:
            if not (stat_cat_month.iloc[stat_cat_month.index.get_loc(index_exp2[-1])].isna()).all():
                continue
            else:
                label.set_color('red')
                label.set_fontweight('bold')
                label.set_fontsize(9.5)
        elif i == x2_2[-1]:
            if (stat_cat_month.iloc[stat_cat_month.index.get_loc(index_exp2[-1])].isna()).all():
                continue
            else:
                label.set_color('red')
                label.set_fontweight('bold')
                label.set_fontsize(9.5)
        else:
            label.set_color('red')
            label.set_fontweight('bold')
            label.set_fontsize(9.5)
        
plt.tight_layout(w_pad=2.5) # -налаштування горизонтального відступу між графіками
plt.ioff()

# 5c. Роздрукуємо статистичні показники для аналізу кожної Категорії та зробимо висновки:
print('5.1 КАТЕГОРІАЛЬНИЙ ПРОФІЛЬ МІСЯЦЯ:\n')
pd.set_option('display.width', None)
print(stat_cat_month, '\n')
print('5.2 ВІЗУАЛІЗАЦІЇ:\n')
print("-5.2.1 'Медіана vs Середнє категорій Доходів'")
print("-5.2.2 'Медіана vs Середнє категорій Витрат'")
print("-5.2.3 'Частотні показники категорій Витрат'\n")
plt.show()
print('5.3 ВИСНОВКИ:\n')
print('1) Основна категорія Доходу -', stat_cat_month['median'].idxmax(), '. Типовий розмір -', int(stat_cat_month.iat[0, 0]), 'грн/міс, що складався зазвичай з', int(stat_cat_month.iat[0, 2]), 'транзакцій. Зарплатня надходила майже кожен місяць.')
print('2) Друга категорія - other_income має інший порядок значень:', int(stat_cat_month.iat[1, 5]), '-',  int(stat_cat_month.iat[1, 6]), 'грн/міс та низьку стабільність:', round(stat_cat_month.iat[1, 4], 1), '. Вона складала лише', int(stat_cat_month.iat[1, 1]), '% від всіх Доходів.')
print('   Cумісне дослідження обох видів Надходжень на більш низьких рівнях агрегації, може значно спотворити їх основні статистичні показники!')
print('3) Найбільш значущі категорії Витрат:' , ', '.join(list(stat_cat_month.iloc[2:, 0:1].nlargest(2, 'median').index)), '- їх місячні значення були найбільшими та разом складали:', stat_cat_month.loc[['food', 'apartment', 'travel'], ['share_med_%']].sum().values.item(), '% всіх Розходів.')
print('4) Найстабільніші категорії, які можна спрогнозувати:' , ', '.join(list(stat_cat_month.iloc[:, 4:5].nsmallest(2, 'cv').index)), '- в цілому мали доволі вузький діапазон можливих показників.')
print('5) Найбільш рідкісні та хаотичні категорії:' , ', '.join(list(stat_cat_month.iloc[:, -1:].nsmallest(2, 'appear_%').index)), '- витрати відбувались не часто, але з великою кількістю екстремальних значень.')
print('6) Найчастіше гроші витрачалися на:' , ', '.join(list(stat_cat_month.iloc[2:, 2:3].nlargest(2, 'activity_med').index)), '- активність цих категорій була щомісячна та складала:', int(stat_cat_month.at['travel', 'activity_med']), 'та', int(stat_cat_month.at['food', 'activity_med']), 'траз/міс відповідно.')
print('-  Кожна категорія містить нетипові екстремальні значення (значний Варіаційний розмах: "max" - "min", порівняно з "median"). Це погіршує')
print('   стабільність даних, але в цілому є нормальним явищем у фінансовій поведінці фізичних осіб.')
print('-  У всіх категоріях спостерігається значна неоднорідність даних (високий показник "cv"). Під час прогнозування особливо нестабільних категорій,')
print('   бажано орієнтуватися на стат. характеристики, які більш стійкі до викидів.')
print('-  Всі дані мають асиметричний розподіл з ознаками "правобічного" перекосу. Доволі ймовірно, що більшість подій носили дрібний характер і лише') 
print('   деякі надходження/витрати були значними (майже усюди "median" << "mean"; "min" < "P25", тоді як "P75" << "max").\n\n')

In [ ]:
# "НАЙКРАЩІЙ" VS "НАЙГІРШИЙ" МІСЯЦЬ

In [ ]:
# 6a. Визначимо найбільш Прибутковий та найбільш Збитковий Місяць з точки зору загальних показнків бюджету.

# 1) виокремимо значення необхідних показників бюджету:
budget_val_mo = budget_month[['year', 'inc_month', 'exp_month', 'profit']].copy() # -використаємо "Агреговану таблицю Надходжень/Витрат за кажен Місяць" (п. 2d)

# 2) порахуємо "Норму заощаджень" для кожного місяця:
budget_val_mo['saving_rate'] = 1 - budget_val_mo['exp_month']/budget_val_mo['inc_month']

# 3) розрахуємо 12-місячні ковзні середні Доходів, Витрат та Прибутку для подальшої побудови Довгострокового фінансового тренду
# -кожне значення тренду в момент t описує попередні 12 місяців, включаючи місяць t. Тобто ковзна середня запізнюється 
# -наприклад, ділянка кривої тренду "липень 2014 - червень 2015" фактично описує динаміку витрат за період: "серпень 2013 - червень 2015"
iep_month = budget_val_mo[['year','inc_month', 'exp_month', 'profit']].copy()

# створимо лист, що містить упорядковані унікальні роки обраного періоду:
years_list = iep_month['year'].tolist()
years_list = sorted(list(set(years_list)))
# стовримо копію у форматі 'str':
list_year = iep_month['year'].astype(str).str[-2:].tolist()
# створюємо більш короткий варінат індексів:
rename_mapping = list()
for idx in iep_month.index:   
    rename_mapping.append(f"{idx[:3]}")
# додаємо їх до кожного значення зі списку
rename_index = [f"{list_val}_{digits}" for list_val, digits in zip(rename_mapping, list_year)]
iep_month = iep_month.set_axis(rename_index, axis='index')
#
trend_iep = iep_month.rolling(12).mean() # розраховуємо 12-місячні ковзні середні

# 4) "Медіанне місячне значення Надходжень Витрат Прибутку та Норми заощаджень":
budget_mo_med = round(budget_val_mo.groupby(budget_val_mo.index, sort=False).agg(['median']), 2)
# видаляємо непотрібний рівень назви стовбчиків та робимо копію даних:
budget_mo_med.columns = budget_mo_med.columns.droplevel(1)
budget_mo = budget_mo_med.copy()

# 5) "10 і 90 Перцентилі" місячних Надходжень та Витрат, щоб нівелювати вплив екстремальних значень на аналіз:
budget_mo[['inc_p10', 'exp_p10']] = budget_val_mo[['inc_month', 'exp_month']].groupby(budget_month.index, sort=False).quantile(0.1)
budget_mo[['inc_p90', 'exp_p90']] = budget_val_mo[['inc_month', 'exp_month']].groupby(budget_month.index, sort=False).quantile(0.9)

# 6) виконаємо форматування зведених сатистичних даних для їх більш наочного представлення:
budget_mo_ = budget_mo.reindex(columns=['inc_month', 'inc_p10', 'inc_p90', 'exp_month', 
                                      'exp_p10', 'exp_p90', 'profit', 'saving_rate'])
#
budget_mo_['saving_rate'] = budget_mo_['saving_rate']*100
#
budget_mo_.rename(columns={'inc_month': 'Надходження_med:', 'inc_p10': 'Надходження_P10:', 'inc_p90': 'Надходження_P90:',
                         'exp_month': 'Витрати_med:','exp_p10': 'Витрати_P10:', 'exp_p90': 'Витрати_P90:',
                         'profit': 'Прибуток_med:', 'saving_rate': 'Норма заощаджень_%:'}, inplace=True)

# 7) знайдемо індекси Найприбутковішого та Найвитратнішого місяця, виконуючи порівняння за Прибутком:
best_month_name = budget_mo_['Прибуток_med:'].idxmax() # -найприбутковіший місяць.
worst_month_name = budget_mo_['Прибуток_med:'].idxmin() # -найвитратніший місяць.

# 8) отримаємо статистичні показники відповідно знайдених індексів:  
best_month_stat = budget_mo_.loc[best_month_name]
worst_month_stat = budget_mo_.loc[worst_month_name]

In [ ]:
# 6b. Побудуємо графіки на основі розрахованих показників для Місячного аналізу

# Вхідні дані:
trend_iep_cut = trend_iep.iloc[11:, :]
#
x = trend_iep_cut.index
y1 = trend_iep_cut['inc_month'].values
y2 = trend_iep_cut['exp_month'].values
y3 = (trend_iep_cut['profit']/trend_iep_cut['inc_month']*100).values # -значення Прибутку у відсотках

# графік Ковзного середнього створюємо, якщо період містить більше 1 року!!!
if len(years_list) > 1:
    # Перший графік:
    fig, ax1 = plt.subplots(figsize=(15, 4)) # -створення фігури та осі з заданими параметрами розміру графіків    
    ax1.bar(x, y3, label='Прибуток', color='orange',  alpha=0.8) # -параметри візуалізації Прибутку
    ax1_1 = ax1.twinx()
    ax1_1.plot(x, y1, label='Доходи', color='green', linewidth=3) # -параметри візуалізації Доходів
    ax1_1.plot(x, y2, label='Витрати', color='magenta', linewidth=3) # -параметри візуалізації Витрат
    # Оформлення:
    ax1.set_title('6.2.1 Ковзне середнє показників Бюджету (12 міс.)') 
    ax1.set_ylabel('Прибуток_%') # -підпис основної вісі "y"
    ax1.set_yticks(np.arange(-30, 20, 5)) # -діапазон та шаг значень вісі "х"
    ax1.tick_params(axis='x', labelrotation=55) # -нахил значень вісі "х"
    ax1.legend(loc='lower left') # -показати зліва знизу легенду графіка, що належить лівій вісі "y"
    ax1_1.legend(loc='lower right') # -показати справа знизу легенду графіка, що належить правій вісі "y"
    ax1_1.set_ylabel('Доходи-Витрати_грн') # -підпис додаткової вісі "y"
    ax1.grid(linestyle='--', linewidth=0.35) 

# підготуємо дані для побудови Другого графіку:
inc_exp_prof = budget_mo_[['Надходження_med:', 'Витрати_med:', 'Прибуток_med:']]
inc_exp_prof = inc_exp_prof.rename(columns={'Надходження_med:': 'Доходи', 'Витрати_med:': 'Витрати', 'Прибуток_med:': 'Прибуток'})
# змінемо значення індексів на більш короткі назви місяців:
short_name_month = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
inc_exp_prof = inc_exp_prof.set_axis(short_name_month, axis='index')
# Другий графік:
inc_exp_prof.plot.bar(rot=0, figsize=(15, 4), color={'Доходи': 'green', 'Витрати': 'magenta', 'Прибуток': 'orange'}, zorder=2)
# Оформлення:
plt.title('6.2.2 Медіана місячних показників Бюджету')
plt.ylabel('Медіанні значення_грн')
plt.grid(linestyle='--', linewidth=0.5)
plt.legend(fontsize=9)
plt.ioff() 

# 6c. Роздрукуємо отримані характеристики та графіки для опису і порівняння показників Бюджету кожного місяця
print('6.1 НАЙПРИБУТКОВІШИЙ vs НАЙВИТРАТНІШИЙ МІСЯЦЬ:\n')
print(budget_mo_.loc[[best_month_name, worst_month_name]].T,'\n')
print('6.2 ВІЗУАЛІЗАЦІЇ:\n')
print("-6.2.1 'Ковзне середнє показників Бюджету (12 міс.) - відображається, якщо у періоді ≥ 2 роки'")
print("-6.2.2 'Медіана місячних показників Бюджету'\n")
plt.show()
print('6.3 ВИСНОВКИ:\n')
print('1) Найприбутковішим місяцем виявився -', best_month_name, '. Він мав Надходження -', int(best_month_stat['Надходження_med:']), 'грн/міс, з високою варіативністю можливих значень:', int(best_month_stat['Надходження_P10:']), '-', int(best_month_stat['Надходження_P90:']), 'грн/міс.')
print('   Витрати у', best_month_name, 'становили -', int(best_month_stat['Витрати_med:']), 'грн/міс та зазвичай носили однорідний характер. Типове значення Прибутку -' , int(best_month_stat['Прибуток_med:']), 'грн/міс, що в перспективі')
print('   дозволяло Заощадити значну частину надходжень -', best_month_stat['Норма заощаджень_%:'], '%.')
print('2) У найвитратніший місяць -', worst_month_name, ', Надходило лише:', int(worst_month_stat['Надходження_P10:']), '-', int(worst_month_stat['Надходження_P90:']), 'грн/міс, а типові Розходи були дуже нестабільні, з Медіаною -', int(worst_month_stat['Витрати_med:']), 'грн/міс.')
print('   Дефіцит', worst_month_name, 'складав близько: (' , int(worst_month_stat['Прибуток_med:']), ') грн/міс. Найнижчий показник Норми заощаджень свідчить про концентрацію високих Витрат в цьому місяці.')
if len(years_list) == 5:
    print('3) Бачимо, що найвитратніший місяць зазвичай наступав одразу після найбільш прибуткового і такий феномен траплявся саме влітку. Це може свідчити')
    print('   про те, що Червень - це місяць перед відпусткою, коли нараховувалися додаткові кошти, а Липень - період віпочинку, що спричиняв додтакові витрати.')
if len(years_list) > 1:
    print('-  На графіку "12-місячних Ковзних середніх" можна виділити наступні періоди, які демонструють згладжену динаміку змін у показниках бюджету:')
    for year in years_list:
        if year == 2013:  
            print('   -"Спокійний період": містив витрані та прибуткові місяці і не мав яскраво вираженої тенденції. Але саме тоді склалося первинне уявлення')
            print('     про розподіл грошей, та спроби спрогнозувати фінансове становище у майбутньому.')
        if year == 2014:        
            print('   -"Поступовий спад": фаза хиткої рівноваги переходить у затяжну негативну тенденцію (можливо через фінансові труднощі та/або початок')
            print('     бойових дій на Донбасі). Згодом з\'являється спроможність до тривалих Заощаджень, що говорить про поступове покращення фінансового здоров\'я.')
        if year == 2015: 
            print('   -"Період стабілізації": ковзна середня Доходів демонструє відновлення, а Витрати зростають більш повільно. Прибуток має позитивну динаміку.')
        if year == 2016:
            print('   -"Тимчасова криза переходить в період Росту": спостерігаються зміни в поведінці показників бюджету. Період Дефіциту характеризується стрімким')
            print('     падінням Доходів на фоні збільшення Витрат. Таке погіршення м.б. спричинено нестабільним Заробітком, додатковими Розходами або високою Інфляцією.')
            print('     Загальна тенденція демонструє зростання досліджувальних характеристик та сигналізує про структурне покращення фінансового балансу і рівня життя.')
print('-  На графіку "Медіана місячних показників Бюджету" бачимо, що зазвичай найприбутковіші місяці це -', ', '.join(list(budget_mo_.iloc[:, 6:7].nlargest(3, 'Прибуток_med:').index)),', а найвитратнішими')
print('   часто стають -', ', '.join(list(budget_mo_.iloc[:, 6:7].nsmallest(3, 'Прибуток_med:').index)),'.\n\n')

In [ ]:
# "ЕКОНОМНИЙ" VS "ВИТРАТНИЙ" ДЕНЬ (МІСЯЦЯ)

In [ ]:
# 7a. Визначимо у якому місяці типовий День виявився найбільш Економним, а у якому - найбільш Витратним.

# ЗАУВАЖЕННЯ: Сумісний статистичний аналіз категорій Надходжень на рівні "День" - є недоцільним і може привести до хибних висновків!

# 1) "Медіана щоденних Витрат" у кожному місяці:
exp_day = day.iloc[:, 2:-1] # -використаємо "Проміжну таблицю Транзакцій" за весь період (п. 2a)
exp_sum_day = (exp_day.iloc[:, :-1].sum(axis=1)).rename('exp_sum_day')
exp_day_c = pd.concat([exp_day['month'], exp_sum_day], axis=1)
exp_day_c['exp_sum_day'] = exp_day_c['exp_sum_day'].astype('float64')
exp_day_median = exp_day_c.groupby('month', sort=False)['exp_sum_day'].median().rename('Медіана_грн:')

# 2) "1 та 3 Квартилі" місячних Витрат, для розуміння розмаху IQR:
exp_day_p25 = exp_day_c.groupby('month', sort=False)['exp_sum_day'].quantile(0.25).rename('P25_грн:')
exp_day_p75 = exp_day_c.groupby('month', sort=False)['exp_sum_day'].quantile(0.75).rename('P75_грн:')

# 3) "IQR" місячних Витрат:
iqr = (exp_day_p75 - exp_day_p25).rename('IQR_грн:')

# 4) "Частка нульових Витратних днів" у кожному місяці:
exp_0d_count = expenses_active[['month','zero_exp_days']].groupby('month', sort=False)['zero_exp_days'].sum() # -використаємо "Проміжну таблицю Надходжень/Витрат" (п. 2d)
day_count = act_mo['month'].value_counts() # -використаємо "Проміжну таблицю Активності" (п. 2c)
share_exp_0d = round(exp_0d_count.div(day_count)*100, 2).rename('"Тихі" дні_%:')

# 5) "Частка днів, де Витрати перевищили 75%" усіх щоденних спостережень:
exp_day_c1 = pd.concat([day['month'], exp_sum_day], axis=1)
ed_more_p75 = exp_day_c1[exp_day_c1['exp_sum_day'] > exp_day_c1['exp_sum_day'].quantile(0.75)]
em_more_p75_count = ed_more_p75.groupby(['month'], sort=False)['exp_sum_day'].count()
share_exp_d_more75 = round(em_more_p75_count.div(day_count)*100, 2).rename('Значні витрати_%:')

# 6) "CQV" Коефіцієнт квартильної варіації Витрат - це відносний та стійкий до викидів показник, 
# який використовується для оцінки ступеня розсіяності (розкиду) даних навколо медіани:
cqv = round((exp_day_p75 - exp_day_p25)/(exp_day_p75 + exp_day_p25)*100, 2).rename('CQV витрат_%:')

# 7) поєднаємо усі розраховані показники у спільну таблицю характеристик Типового дня у кожному Місяці:
stat_exp_day = pd.concat([exp_day_median, exp_day_p25, exp_day_p75, iqr, share_exp_0d, share_exp_d_more75, cqv], axis=1)
stat_exp_day.index.name = None

# 8) знайдемо індекси місяців, де типовий День виявився Економним, а де - Витратним:
best_mo_d_name = stat_exp_day['Медіана_грн:'].idxmin() # -найбільш Економний день.
worst_mo_d_name = stat_exp_day['Медіана_грн:'].idxmax() # -найбільш Витратний день.

# 9) отримаємо статистичні показники відповідно знайдених індексів:  
best_mo_d_stat = stat_exp_day.loc[best_mo_d_name]
worst_mo_d_stat = stat_exp_day.loc[worst_mo_d_name]

In [ ]:
# 7b. Побудуємо графіки на основі розрахованих показників для аналізу типового Дня у кожному місяці

# змінемо значення індексів на більш короткі назви місяців:
short_name_month = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
stat_exp_day_short = stat_exp_day.set_axis(short_name_month, axis='index')

# Вхідні дані:
x = stat_exp_day_short.index
y1 = stat_exp_day_short['P25_грн:']; y2 = stat_exp_day_short['Медіана_грн:']
y3 = stat_exp_day_short['P75_грн:']; y4 = stat_exp_day_short['"Тихі" дні_%:']
y5 = stat_exp_day_short['Значні витрати_%:']; y6 = stat_exp_day_short['CQV витрат_%:']

plt.figure(figsize=(15, 4))

# Перший графік:
plt.subplot(1, 2, 1)
plt.plot(x, y1, label='P25/P75', color='black', alpha=0.8) # -параметри візуалізації 25 перцентиля Витрат
plt.plot(x, y2, label='Median', color='magenta', linewidth=2) # -параметри візуалізації медіани Витрат
plt.plot(x, y3, color='black', alpha=0.8) # -параметри візуалізації 75 перцентиля Витрат
plt.fill_between(x, y1, y3, label='IQR', color='gray', alpha=0.2) # -параметри візуалізації міжквартильного діапазону Витрат
# Оформлення:
plt.title('7.2.1 Абсолютна мінливість Розходів')
plt.ylabel("Діапазон витрат_грн") 
plt.grid(linestyle='--', linewidth=0.5) 
plt.legend() 

# Другий графік:
plt.subplot(1, 2, 2)
plt.bar(x, y6, label='CQV', color='tab:blue', zorder=2) # -параметри візуалізації Коефіцієнта квартильної варіації Витрат
# Оформлення:
plt.title('7.2.2 Відносна мінливість Розходів')
plt.ylabel("Ступінь розсіяності_%")
plt.grid(linestyle='--', linewidth=0.5)
plt.legend()
plt.ioff() 

# 7c. Роздрукуємо отримані характеристики і графіки для опису та порівняння Витрат Дня у кожному місяці

print('7.1 НАЙЕКОНОМІШНИЙ vs НАЙВИТРАТНІШИЙ ДЕНЬ:\n')
print(stat_exp_day.loc[[best_mo_d_name, worst_mo_d_name]].T, '\n')
print('7.2 ВІЗУАЛІЗАЦІЇ:\n')
print("-7.2.1 'Абсолютна мінливість Розходів'")
print("-7.2.2 'Відносна мінливість Розходів'\n")
plt.show()
print('7.3 ВИСНОВКИ:\n')
print('1) Найдешевший день виявився типовим саме для', best_mo_d_name, ', з Розходами -', int(best_mo_d_stat['Медіана_грн:']), 'грн/день. У більшості випадків Витрати були дрібними, та коливались у')
print('   діапазоні:', int(best_mo_d_stat['P25_грн:']), '-', int(best_mo_d_stat['P75_грн:']), 'грн/день, а їх', int(round(100/best_mo_d_stat['Значні витрати_%:'], 0)), 'частина мала аномально високі значення відносно медіанного рівня.',best_mo_d_name, 'мав', round(best_mo_d_stat['"Тихі" дні_%:']), '% днів без Розходів.')
print('2) Найбільш витратний день видався у', worst_mo_d_name, '. Тут Розходи були найменш контрольованими:', int(worst_mo_d_stat['P25_грн:']), '-', int(worst_mo_d_stat['P75_грн:']), 'грн/день, а їх типовий рівень в', round(worst_mo_d_stat['Медіана_грн:']/best_mo_d_stat['Медіана_грн:'], 1), 'рази')    
print('   вищій ніж у', best_mo_d_name, '(', int(worst_mo_d_stat['Медіана_грн:']), 'грн/день ). У найдорожчому місяці - був всього 1 "тихий" день,', int(round(worst_mo_d_stat['Значні витрати_%:'] ,0)), '% витрат', worst_mo_d_name, 'мали непомірно високі значення.')
print('   Високий розмір та імпульсивність розходів може пояснюватися можливою відпусткою та/або святковими заходами у цей період.')
print('-  Розташування медіани відносно меж IQR показує, що протягом всього періоду більшість денних Витрат носили дрібний або помірний характер.')
print('-  Однорідні денні розходи мають місяці —', ', '.join(list(stat_exp_day.iloc[:, 3:4].nsmallest(2, 'IQR_грн:').index)), '. Найвища абсолютна варіативність даних (IQR) спостерігається у:', ' та '.join(list(stat_exp_day.iloc[:, 3:4].nlargest(2, 'IQR_грн:').index)), '.')
print('-  Найменш "ризикові" місяці відносно рівня життя —', ', '.join(list(stat_exp_day.iloc[:, -1:].nsmallest(2, 'CQV витрат_%:').index)), '. Найбільша ймовірність денних Перевитрат може статися у:', ' та '.join(list(stat_exp_day.iloc[:, -1:].nlargest(2, 'CQV витрат_%:').index)), '.\n\n')

In [ ]:
# "ЕКОНОМНА" VS "ВИТРАТНА" ТРАНЗАКЦІЯ (МІСЯЦЯ)

In [ ]:
# 8a. Визначимо у якому місяці Транзакції мають найбільш Прибутковий/Економний характер, а у якому - Незначний/Витратний.

# ЗАУВАЖЕННЯ: для коректного дослідження категорій Надходжень на рівні "Транзакції" їх хар-ки розрахуємо та проаналізуємо окремо!

# 1) "Медіанна транзакція Зарпалтні - salary":
salary_trans = day[['month', 'salary']] # -використаємо "Проміжну таблицю Транзакцій" за весь період (п. 2a)
salary_trans_med = salary_trans.groupby('month', sort=False)['salary'].median()
salary_trans_med.rename('salary_med:', inplace=True)

# 2) "Відсоток днів з транзакціями категорії salary":
salary_act = act_mo[['month', 'salary']] # -використаємо "Проміжну таблицю Активності" за весь період (п. 2c)
salary_act_count = salary_act.groupby('month', sort=False)['salary'].sum()
days_with_salary = round(salary_act_count.div(day_count)*100, 2).rename('Частка salary_%:')

# 3) "Медіанна транзакція Інших доходів - other_income":
other_inc_trans = day[['month', 'other_income']]
other_inc_trans_med = other_inc_trans.groupby('month', sort=False)['other_income'].median()
other_inc_trans_med.rename('other_income_med:', inplace=True)

# 4) "Відсоток днів з транзакціями категорії 'other_income":
other_inc_act = act_mo[['month', 'other_income']]
other_inc_act_count = other_inc_act.groupby('month', sort=False)['other_income'].sum()
days_with_other_inc = round(other_inc_act_count.div(day_count)*100, 2).rename('Частка other_income_%:')

# 5) "Медіанне значення усіх тразнакцій Витрат":
exp_trans = day.iloc[:, 2:-1]
exp_trans_melted = exp_trans.melt(id_vars=['month'], value_vars=exp_trans.iloc[:, :-1], value_name='Витрати_med:')
exp_trans_med = exp_trans_melted.groupby('month', sort=False)['Витрати_med:'].median()

# 6) "Медіанне значення щоденної Кількості Витратних транзакцій":
exp_act_sum = act_mo.iloc[:, 2:-2].sum(axis=1)
exp_txn_p_day = pd.concat([act_mo['month'], exp_act_sum.rename('К-ть Витрат_med:')], axis=1)
exp_txn_p_day_med = exp_txn_p_day.groupby('month', sort=False)['К-ть Витрат_med:'].median()

# 7) "Частка Імпульсивних витратних транзакцій":
def exp_txn_mo(x):
    """
    Input: "x" - кожне значення усіх стовбців таблиці з Транзакціями Витрат
    Return: "x > limit" - поточне числове значення у стовбці, якщо воно перевищує: 'limit'
            "x" - кожне значення усіх стовбців, що мають тип: 'object' 
    """
    if x.dtype == 'float64':
        q25 = x.quantile(0.25)
        q75 = x.quantile(0.75)
        limit = q75 + 1.5 * (q75 - q25)
        return x.where (x > limit) 
    else:
        return x
exp_imp_day = exp_trans.apply(exp_txn_mo)
# 
exp_imp_mo = exp_imp_day.groupby('month', sort=False).count()
exp_imp_mo_sum = exp_imp_mo.sum(axis=1) 
#
exp_act_day = act_mo.iloc[:, 2:-1]
exp_act_cat_mo = exp_act_day.groupby('month', sort=False).sum()
exp_act_sum_mo = exp_act_cat_mo.sum(axis=1)
#
exp_imp_txn_mo = round(exp_imp_mo_sum.div(exp_act_sum_mo)*100, 2).rename('Частка Імп.витрат_%:')

# 8) поєднаємо усі розраховані показники у спільну таблицю поведінки Транзакцій у кожному місяці:
stat_trans_lvl = pd.concat([exp_trans_med, exp_txn_p_day_med, exp_imp_txn_mo, salary_trans_med, 
                            days_with_salary, other_inc_trans_med, days_with_other_inc], axis=1)
stat_trans_lvl.index.name = None

# 9) Знайдемо місяці, де Транзакції виявилися Прибутковими/Економними, а де - Незначними/Витратними:
best_mo_t_name = stat_trans_lvl['Витрати_med:'].idxmin() # -найбільш Прибуткові/Економні транзацкії.
worst_mo_t_name = stat_trans_lvl['Витрати_med:'].idxmax() # -найбільш Незначні/Витратні транзацкії.

# 10) Отримаємо статистичні показники відповідно знайдених індексів:  
best_mo_t_stat = stat_trans_lvl.loc[best_mo_t_name]
worst_mo_t_stat = stat_trans_lvl.loc[worst_mo_t_name]

In [ ]:
# 8b. Побудуємо графіки на основі розрахованих показників для аналізу Транзакцій у кожному місяці

# змінемо значення індексів на більш короткі назви місяців:
short_name_month = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
stat_trans_lvl_short = stat_trans_lvl.set_axis(short_name_month, axis='index')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5)) # -створення фігури та двой осей з заданими параметрами розміру графіків

# Вхідні дані:
x = stat_trans_lvl_short.index
y1 = stat_trans_lvl_short['salary_med:']; y2 = stat_trans_lvl_short['Частка salary_%:']
y3 = stat_trans_lvl_short['other_income_med:']; y4 = stat_trans_lvl_short['Частка other_income_%:']
y5 = stat_trans_lvl_short['Витрати_med:']; y6 = stat_trans_lvl_short['К-ть Витрат_med:']
y7 = stat_trans_lvl_short['Частка Імп.витрат_%:'];

# Перший графік:
ax1.plot(x, y1, label='Зарплата', linewidth=2, color='green') 
ax1_1 = ax1.twinx()
ax1_1.plot(x, y5, label='Витрати', linewidth=2, color='magenta') 
# Оформлення:
ax1.set_title('8.2.1 Дохід vs Розхід') 
ax1.grid(linestyle='--', linewidth=0.5)
ax1.set_ylabel('Зарплата_грн')
ax1.legend(loc='upper left', fontsize='small') 
ax1_1.set_ylabel('Витрати_грн')
ax1_1.legend(loc='upper right', fontsize='small') 

# Другий графік:
width = 0.35  # -ширина стовбця
x_num = np.arange(len(x))
subbar1 = ax2.bar(x_num - width/2, y6, width, label='К-ть витрат_med', color='tab:blue', zorder=2) # -параметри Першої візуалізації
subbar2 = ax2.bar(x_num + width/2, y7, width, label='Частка Імп.витрат_%', color='orange', zorder=2) # -параметри Другої візуалізації
# Оформлення:
ax2.set_title('8.2.2 Об\'єм витрат')
ax2.set_ylabel('Показники витрат')
ax2.set_xticks(x_num)
ax2.set_xticklabels(x)
ax2.legend(fontsize='small')
ax2.grid(linestyle='--', linewidth=0.5)
plt.tight_layout(w_pad=2.5) # -налаштування горизонтального відступу між графіками
plt.ioff()

# 8c. Роздрукуємо отримані характеристики та графіки для опису та порівняння Транзакцій у кожному місяці
print('8.1 НАЙЕКОНОМНІША vs НАЙВИТРАТНІША ТРАНЗАКЦІЯ:\n')
print(stat_trans_lvl.loc[[best_mo_t_name, worst_mo_t_name]].T, '\n')
print('8.2 ВІЗУАЛІЗАЦІЇ:\n')
print("-8.2.1 'Дохід vs Розхід'")
print("-8.2.2 'Об\'єм витрат'\n")
plt.show()
print('8.3 ВИСНОВКИ:\n')
print('1) У', best_mo_t_name, 'одноразова Витрата була Найменшою за рік, величиною -', int(round(best_mo_t_stat['Витрати_med:'],0)),'грн. За день кошти зазвичай витрачались на', int(best_mo_t_stat['К-ть Витрат_med:']), 'категорії та мали лише', int(round(best_mo_t_stat['Частка Імп.витрат_%:'], 0)), '%')
print('   Імпульсивних розходів. У такий місяць нарахування Зарплатні складало', int(best_mo_t_stat['salary_med:']), 'грн/транз, з приблизною частотою', int(round(30/100*best_mo_t_stat['Частка salary_%:'], 0)), 'раз(и) на місяць.')
print('2) Медіана Розходів у', worst_mo_t_name, 'досягала свого максимуму -', int(worst_mo_t_stat['Витрати_med:']), 'грн/транз та в', round(worst_mo_t_stat['Витрати_med:']/best_mo_t_stat['Витрати_med:'], 1), 'раз(и) перебільшила аналогічний показник', best_mo_t_name, '. Такі витрати')
print('   траплялися', int(worst_mo_t_stat['К-ть Витрат_med:']), 'раз(и) на день,', round(worst_mo_t_stat['Частка Імп.витрат_%:'], 1), '% з яких були аномально високими. Зарплатні платежі складали -', int(worst_mo_t_stat['salary_med:']), 'грн/транз і відбувались', int(round(30/100*worst_mo_t_stat['Частка salary_%:'], 0)), 'раз(и)/міс.')
print('3) Отже,', best_mo_t_name, '- найприбутковіший за рахунок підвищених показників зарплатні та менших чеків, тоді як', worst_mo_t_name, '- містить найбільш збиткові')
print('   транзакції внаслідок зменшеного рівня доходів та більшої кількості емоційних витрат.')
print('-  На 8.2.1 прослідковується певний взаємозв\'язок між кривими Доходів та Розходів - їх характер або співпадає, або - розходи повторюють поведніку дохоів')
print('   із запізненням в 1 місяць. Ймовірно, що Витрати прогнозувалися Зарплатою: у першому випадку спостерігалося їх "миттєве споживання", а у другому')
print('   - розмір розходів залежив від доходів попереднього місяця.', stat_trans_lvl_short['salary_med:'].idxmax(), 'мав найбільше разове Надходження, а', stat_trans_lvl_short['Витрати_med:'].idxmax(), '- максимальну Витратну транзакцію.')
print('-  На 8.2.2 бачимо, що візуалізація частки Імпульсивних витрат має хвилеподібну форму, що вказує на здатність часткового контролю нетипових розходів')
print('   після їх зростання. Витарти', stat_trans_lvl_short['Частка Імп.витрат_%:'].idxmax(), 'містили найбільшу Частину значних імпульсивних транзакцій, тоді як у', stat_trans_lvl_short['Частка Імп.витрат_%:'].idxmin(), 'вона була мінімальна. Денні Витрати')
print('   майже завжди складалися з 2 категорій - це може свідчити про наявність структури у щоденній фінансовій поведінці.\n\n')

In [ ]:
# АНАЛІЗ ДАНИХ НА РІВНІ - "ДНІ ТИЖНЯ"

In [ ]:
# 9a. Розрахуємо необхідні статистичні характеристики для аналізу кожного дня Тижня

# 1) підготовка даних для подальшого аналізу:
period_week = period_c.drop(['record_date'], axis=1) # -використаємо "Вхідну таблицю денних Тразакцій" (п. 1b)
# визначаємо необхідний порядок днів тижня:
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
# перетворимо стовбчик 'weekday' на категоріальний тип із заданим порядком:
period_week['weekday'] = pd.Categorical(period_week['weekday'], categories=day_order, ordered=True)
#
# виокремимо та порахуємо сумарні значення категорій Доходів/Витрат за кожен день:
inc_weekday = period_week.iloc[:, :3]
inc_weekday['Доходи:'] = inc_weekday.iloc[:, 1:].sum(axis=1)
inc_weekday_NaN = inc_weekday.replace(0, np.nan)
#
exp_weekday = pd.concat([period_week.iloc[:, :1], period_week.iloc[:, 3:]], axis=1)
exp_weekday['Витрати:'] = exp_weekday.iloc[:, 1:].sum(axis=1)
exp_weekday_NaN = exp_weekday.replace(0, np.nan)

# 2) "Сумарні значення Надходжень/Витрат по днях тижня":
inc_sum_week = inc_weekday.groupby('weekday', observed=True).sum()
exp_sum_week = exp_weekday.groupby('weekday', observed=True).sum()

# 3) "Відсоток кожної категорії від загальної суми Надходжень/Витрат":
inc_share_week = round(inc_sum_week.iloc[:, :-1].div(inc_sum_week['Доходи:'], axis=0) *100, 2)
exp_share_week = round(exp_sum_week.iloc[:, :-1].div(exp_sum_week['Витрати:'], axis=0) *100, 2)

# 4) "Медіанні значення Надходжень/Витрат по днях тижня":
inc_median_week = inc_weekday_NaN.iloc[:, :-1].groupby('weekday', observed=True).median() # -категоріальні значення Доходів по днях тижня
inc_median_week_ = inc_median_week.rename(columns={'salary': 'salary_med:', 'other_income': 'other_income_med:'}).fillna(0) 
#
exp_median_week = exp_weekday_NaN.groupby('weekday', observed=True).median() # -категоріальні + сумарні значення Витрат по днях тижня
exp_median_week_ = exp_median_week.rename(columns={'Витрати:': 'Витрати_med:'}) 

# 5) "Активність категорій Надходжень/Витрат по днях тижня":
inc_count_week = inc_weekday_NaN.iloc[:, :-1].groupby('weekday', observed=True).count()
inc_count_week_ = inc_count_week.rename(columns={'salary': 'К-ть salary:', 'other_income': 'К-ть other_income:'})
inc_act_sum_week = inc_count_week.sum(axis=1).rename('К-ть Доходів:') # -сумарна активність Надходжень
#
exp_count_week = exp_weekday_NaN.iloc[:, :-1].groupby('weekday', observed=True).count()
exp_act_sum_week = exp_count_week.sum(axis=1).rename('К-ть Витрат:') # -сумарна активність Витрат

# 6) "Медіанне значення Кількості Витратних транзакцій по днях тижня":
exp_count_txn_weekday_ = exp_weekday_NaN.iloc[:, 1:-1].count(axis=1)
exp_count_txn_weekday = pd.concat([exp_weekday_NaN['weekday'], exp_count_txn_weekday_.rename('К-ть Витрат_med:')], axis=1)
exp_count_med_weekday = exp_count_txn_weekday.groupby('weekday', observed=True)['К-ть Витрат_med:'].median()

# 7) "Частка днів тижня, коли відбувались транзакції Надходжень":
inc_count_days_week = inc_weekday.iloc[:, :-1].groupby('weekday', observed=True).count()
inc_count_share_week = round(inc_count_week.div(inc_count_week.sum(axis=0), axis=1)*100, 2)
inc_count_share_week_ = inc_count_share_week.rename(columns={'salary': 'Частка salary_%:', 'other_income': 'Частка other_income_%:'})

# 8) "Частка Імпульсивних витратних транзакцій у дні тижня":
def exp_txn_week(x):
    """
    Input: "x" - кожне значення усіх стовбців таблиці з Транзакціями Витрат
    Return: "x > limit" - поточне числове значення у стовбці, якщо воно перевищує: 'limit'
            "x" - кожне значення усіх стовбців, що мають не мають тип: 'float64' 
    """
    if x.dtype == 'float64':
        q25 = x.quantile(0.25)
        q75 = x.quantile(0.75)
        limit = q75 + 1.5 * (q75 - q25)
        return x.where (x > limit) 
    else:
        return x
imp_exp_weekday = exp_weekday_NaN.iloc[:, :-1].apply(exp_txn_week)
#
imp_exp_week = imp_exp_weekday.groupby('weekday', observed=True).count()
imp_exp_week_sum = imp_exp_week.sum(axis=1) 
imp_exp_txn_week = round(imp_exp_week_sum.div(exp_act_sum_week)*100, 2).rename('Частка Імп.витрат_%:')

# 9) "Частка нульових Витратних днів тижня":
exp_count_0_weekday = pd.concat([exp_weekday['weekday'], expenses_active['zero_exp_days']], axis=1)
exp_sum_0_weekday = exp_count_0_weekday.groupby('weekday', observed=True)['zero_exp_days'].sum()
weekday_count = exp_weekday['weekday'].value_counts()
share_exp_0d_week = round(exp_sum_0_weekday.div(weekday_count)*100, 2).rename('"Тихі" дні_%:')

# 10) поєднаємо розраховані показники у спільну таблицю характеристик даних на рівні - "Тиждень":
stat_week_lvl = pd.concat([inc_median_week_['salary_med:'], inc_count_share_week_['Частка salary_%:'], 
                           exp_median_week_['Витрати_med:'], exp_count_med_weekday, imp_exp_txn_week, share_exp_0d_week], axis=1)
stat_week_lvl.index.name = None

In [ ]:
# 9b. Побудуємо графіки на основі розрахованих показників для аналізу днів Тижня

# нормалізуємо значення характеристик тижня від "0" до "1" для наочного представлення їх екстремумів:
stat_week_lvl_norm = stat_week_lvl.apply(lambda x: (x - x.min()) / (x.max() - x.min()))
stat_week_lvl_norm = stat_week_lvl_norm.fillna(0)

# Вхідні дані:
x = stat_week_lvl_norm.index
y1 = stat_week_lvl_norm['salary_med:']; y2 = stat_week_lvl_norm['Частка salary_%:']
y3 = stat_week_lvl_norm['Витрати_med:']; y4 = stat_week_lvl_norm['К-ть Витрат_med:']
y5 = stat_week_lvl_norm['Частка Імп.витрат_%:']; y6 = stat_week_lvl_norm['"Тихі" дні_%:']
# пошук Максимума та Мінімума:
max_x1 = y1.idxmax(); max_y1 = y1.max()
min_x1 = y1.idxmin(); min_y1 = y1.min()
max_x2 = y2.idxmax(); max_y2 = y2.max()
min_x2 = y2.idxmin(); min_y2 = y2.min()
max_x3 = y3.idxmax(); max_y3 = y3.max()
min_x3 = y3.idxmin(); min_y3 = y3.min()
max_x5 = y5.idxmax(); max_y5 = y5.max()
min_x5 = y5.idxmin(); min_y5 = y5.min()
max_x6 = y6.idxmax(); max_y6 = y6.max()
min_x6 = y6.idxmin(); min_y6 = y6.min()

plt.figure(figsize=(14, 3.5))

# Перший графік:
plt.subplot(1, 2, 1)
plt.plot(x, y3, label='Витрати', linewidth=2.25, color='magenta')
plt.plot(x, y2, label='Частка salary', linewidth=2.25, color='tab:blue', zorder=2) # -підпис та колір Візуалізацій
plt.plot(x, y4, label='К-ть витрат', linewidth=2.25, color='purple')
plt.scatter([max_x2], [max_y2], color='green', s=30, zorder=3) # -відображення Екстремумів
plt.scatter([min_x2], [min_y2], color='red', s=30, zorder=3)
plt.scatter([max_x3], [max_y3], color='green', s=30, zorder=3)
plt.scatter([min_x3], [min_y3], color='red', s=30, zorder=3)
plt.title("9.2.1 Екстремуми 'Витрат', 'Частки salary_%', 'К-ті витрат'")
plt.ylabel('Min-Max Scaling') # -підпис осі y
plt.grid(axis='x', linestyle='--', linewidth=0.5) # -наявність сітки по осі x
plt.legend(fontsize='small') # -розмір тексту легенди

# Другий графік:
plt.subplot(1, 2, 2)
plt.plot(x, y6, label='"Тихі" дні', linewidth=2.25, color='grey')
plt.plot(x, y5, label='Імп.витрати', linewidth=2.25, color='orange')
plt.plot(x, y1, label='salary', linewidth=2.25, color='blue')
plt.scatter([max_x1], [max_y1], color='green', s=30, zorder=3)
plt.scatter([min_x1], [min_y1], color='red', s=30, zorder=3)
plt.scatter([max_x5], [max_y5], color='green', s=30, zorder=3)
plt.scatter([min_x5], [min_y5], color='red', s=30, zorder=3)
plt.scatter([max_x6], [max_y6], color='green', s=30, zorder=3)
plt.scatter([min_x6], [min_y6], color='red', s=30, zorder=3)
plt.title("9.2.2 Екстремуми 'Тихих днів', 'Імп.витрат_%', 'salary'")
plt.ylabel('Min-Max Scaling')
plt.grid(axis='x', linestyle='--', linewidth=0.5)
plt.legend(fontsize='small')
plt.ioff() 

# 9c. Роздрукуємо отримані характеристики і графіки для опису та порівняння днів Тижня

print('9.1 АНАЛІЗ ДНІВ ТИЖНЯ:\n')
print(stat_week_lvl.T.convert_dtypes(), '\n')
print('9.2 ВІЗУАЛІЗАЦІЇ:\n')
print("-9.2.1 'Екстремуми Витрат, Частки salary_% та К-ті витрат'")
print("-9.2.2 'Екстремуми Тихих днів, Імп.витрат_% та salary'\n")
plt.show()
print('9.3 ВИСНОВКИ:\n')
print('1) Понеділок - спокійний початок тижня. Покупки відбувалися регулярно, з',"Мінімальною" if stat_week_lvl_norm.iat[0, 4] == 0 else "невеликою",'Часткою аномальних значень -',round(stat_week_lvl.iat[0, 4], 1), '%.',"Найнижча зарплата," if stat_week_lvl_norm.iat[0, 0] == 0 else "Зарплата,")
print('   у розмірі:',int(stat_week_lvl.iat[0, 0]),'грн, надходила у',round(stat_week_lvl.iat[0, 1], 1),'% випадків.',"Витрати були Найменшими та" if stat_week_lvl_norm.iat[0, 2] == 0 else "Витрати",'зазвичай складалися з',int(stat_week_lvl.iat[0, 3]),'транзакцій, на суму',int(round(stat_week_lvl.iat[0, 2], 0)),'грн.')
print('2) з Вівторка по Четверг - проходили звичайні будні. Дохід мав посередній розмір, але надходив з великою ймовірністю. Витрати у цей період')
print('   не відрізнялись значною кількістю Імпульсивних транзакцій, а їх сума була помірною. Також, доволі часто, траплялися дні без витрат.')
print('3) П\'ятниця та Субота - Найвитратніші дні тижня! Тут загальна вартість Розходів завжди була Найбільшою -', int(stat_week_lvl['Витрати_med:'].iloc[4:6].max()), 'грн/день, що характерно')
print('   для періоду відпочинку.',"Максимальна" if stat_week_lvl['salary_med:'].iloc[4:6].max() == stat_week_lvl['salary_med:'].max() else "Висока",'зарплата -',int(stat_week_lvl['salary_med:'].iloc[4:6].max()),'грн',"Найчастіше" if stat_week_lvl['Частка salary_%:'].iloc[4:6].max() == stat_week_lvl['Частка salary_%:'].max() else "дуже часто",'надходила саме в ці дні (',round(stat_week_lvl['Частка salary_%:'].iloc[4:6].max(), 1),'% випадків ), посилюючі бажання витрачати.')
print('   До того ж, Значні показники частки Імпульсивних витрат та ймовірністі "Тихого" дня створювали додатковий ризик Перевитрат в цей період.')
print('4) Неділя - зазвичай Екномний день. Йому були властиві',"Мінімальна" if stat_week_lvl.iat[6, 2] == stat_week_lvl['Витрати_med:'].min() else "невелика",'сума Розходів -',int(stat_week_lvl.iat[6, 2]),'грн/день та спокійний характер транзакцій. Але ймовірність')
print('   отримати Зарплату тут також виявилася',"Найнижчою" if stat_week_lvl.iat[6, 1] == stat_week_lvl['Частка salary_%:'].min() else "Низькою",'-',round(stat_week_lvl.iat[6, 1],1),'%, що можливо і вплинуло на раціональність фінансової поведінки.')
print('-  На графіках, протягом тижня, спостерігається зв\'язок у поведінці деяких показників. Зліва бачимо, що Витрати зростають у ті дні, коли підвищується')
print('   ймовірність отримання Зарплати, а справа часто прослідковується залежність: чим більша Зарплатня - тим більша Частка Імпульсивних покупок.')
print('   Це може свідчити про зниження самоконтролю після отримання Доходу або про планування Розходів під Зарплату. Водночас схожість графіків') 
print('   сама по собі не доводить причинно-слідчий зв\'язок, його наявність треба перевіряти додатково.\n')